# Stage 3: XGBoost Triage Gate, final 42 columns

SimGuard pipeline, Stage 3. This notebook retrains the triage model on the final feature list in `artifacts/xgboost_feature_columns_final.json` (42 columns) and replaces the 53 column model from `xgboost_triage.ipynb`.

**Why the feature list changed**

- `xgboost_ablation.ipynb` trained four variants on 10 repeated 70/15/15 SIM splits. Removing the Prophet columns, the voice/SMS columns, or both changed test average precision only within noise (no metric moved by more than one standard deviation of the paired difference). The variant with both removed (`no_voice_sms_no_prophet`) averaged 0.9265 test AP against 0.9268 for the full 53 columns.
- The Prophet features used by `xgboost_triage.ipynb` (`prophet_features.parquet`) were forecast per billing cycle and broadcast onto every day of that cycle, so a row could see a forecast fitted on days after its own date. That is a look-ahead issue (see `prophet_lookahead_audit.ipynb`).
- So the final model drops the 7 Prophet columns and the 4 raw voice/SMS columns. `voice_utilization_ratio` stays in.

**What is the same as `xgboost_triage.ipynb`**

Seed 42, the same hyperparameters except `early_stopping_rounds` (50 here instead of 20), `scale_pos_weight = negatives / positives` on train, early stopping on validation `aucpr`, the same gate (`< 0.10` auto-clear, `> 0.90` auto-flag, otherwise escalate to Stage 4 CBR), the same evaluation steps.

Why patience 50: on this feature set the validation `aucpr` dips between about tree 11 and tree 21 and peaks at tree 57. With patience 20, training stopped at 11 trees, every probability fell between 0.10 and 0.90, and every row escalated.

**What is different**

1. No Prophet load or merge (old sections 4, 6 and 7 removed).
2. The feature list is read from `artifacts/xgboost_feature_columns_final.json` instead of being every non-id column.
3. No `%pip install` cell. Packages come from `requirements.txt` and `requirements-dev.txt` in the `ml-service/venv`.
4. MLflow experiment `simguard-stage3-xgboost-final` in the root `mlruns.db`.
5. A determinism check fits the model a second time and compares validation predictions.
6. SHAP is limited to the trees that make predictions (`iteration_range=(0, best_iteration + 1)`, 57 trees). The booster also keeps the 50 trees grown after the best one, and `predict_proba` ignores them. The first version of this notebook (and `xgboost_triage.ipynb`) explained all trees, so its SHAP values and base value described a different model from the one that predicts.

The 53 column artifacts are kept in `artifacts/archive_53col/`. Run from `ml-service/` or `ml-service/notebooks/`. No git operations.

## 1. Imports

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import mlflow, mlflow.xgboost
import joblib
import sklearn
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

import sys
warnings.filterwarnings("ignore")
print("python", sys.executable)
print("xgboost", xgb.__version__, "| shap", shap.__version__, "| mlflow", mlflow.__version__,
      "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

C:\Users\ADMIN\Desktop\projects\simguard\ml-service\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/11/02 11:47:38 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\ADMIN\Desktop\projects\simguard\ml-service\venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


python C:\Users\ADMIN\Desktop\projects\simguard\ml-service\venv\Scripts\python.exe
xgboost 3.4.1 | shap 0.52.0 | mlflow 3.16.1 | scikit-learn 1.8.0 | pandas 3.0.0


## 2. Seed, paths, MLflow experiment

In [2]:
SEED = 42
np.random.seed(SEED)

DATA_DIR = "data"
ARTIFACT_DIR = "artifacts"

# If the notebook is started from ml-service/notebooks, step up to ml-service/
if not os.path.isdir(DATA_DIR) and os.path.isdir(os.path.join("..", DATA_DIR)):
    os.chdir("..")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

GATE_LOW, GATE_HIGH = 0.10, 0.90

MLFLOW_DB = "sqlite:///mlruns.db"          # root ml-service/mlruns.db
mlflow.set_tracking_uri(MLFLOW_DB)
EXPERIMENT_NAME = "simguard-stage3-xgboost-final"
mlflow.set_experiment(EXPERIMENT_NAME)

print("CWD          :", os.getcwd())
print("DATA_DIR     :", DATA_DIR, "| exists:", os.path.isdir(DATA_DIR))
print("ARTIFACT_DIR :", ARTIFACT_DIR)
print("Tracking URI :", mlflow.get_tracking_uri())
print("Experiment   :", EXPERIMENT_NAME)

CWD          : C:\Users\ADMIN\Desktop\projects\simguard\ml-service
DATA_DIR     : data | exists: True
ARTIFACT_DIR : artifacts
Tracking URI : sqlite:///mlruns.db
Experiment   : simguard-stage3-xgboost-final


## 3. Load cohort Z-score feature set (X_v2) and labels (y), kept separate

In [3]:
X_train = pd.read_parquet(f"{DATA_DIR}/X_train_v2.parquet")
X_val   = pd.read_parquet(f"{DATA_DIR}/X_val_v2.parquet")
X_test  = pd.read_parquet(f"{DATA_DIR}/X_test_v2.parquet")
y_train = pd.read_parquet(f"{DATA_DIR}/y_train.parquet")
y_val   = pd.read_parquet(f"{DATA_DIR}/y_val.parquet")
y_test  = pd.read_parquet(f"{DATA_DIR}/y_test.parquet")

FORBIDDEN = ["is_anomalous", "misuse_category"]
for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    assert len(X) == len(y), f"{name}: X/y length mismatch"
    assert (X["sim_id"].values == y["sim_id"].values).all() and (X["date"].values == y["date"].values).all(), \
        f"{name}: X and y rows are not aligned on (sim_id, date)"
    assert not any(c in X.columns for c in FORBIDDEN), f"{name}: label column found in X"
    print(f"{name:<5} X={X.shape}  y={y.shape}  anomaly rate={y['is_anomalous'].mean():.4f}  SIMs={X['sim_id'].nunique()}")

train X=(63000, 48)  y=(63000, 4)  anomaly rate=0.0205  SIMs=700
val   X=(13500, 48)  y=(13500, 4)  anomaly rate=0.0388  SIMs=150
test  X=(13500, 48)  y=(13500, 4)  anomaly rate=0.0267  SIMs=150


## 4. Add `billing_cycle_num` (same rule as before)

Not a feature. It is kept only so the escalated SHAP index CSV has the same three columns as before (`sim_id`, `date`, `billing_cycle_num`).

In [4]:
SIM_START = pd.Timestamp("2025-01-01")
EXPECTED_CYCLES = {0, 1, 2}
for name, X in [("train", X_train), ("val", X_val), ("test", X_test)]:
    day_offset = (X["date"] - SIM_START).dt.days
    X["billing_cycle_num"] = (day_offset // 30).astype(int)
    seen = set(X["billing_cycle_num"].unique())
    print(f"{name:<5} dates {X['date'].min().date()} to {X['date'].max().date()} | cycles {sorted(seen)}")
    assert seen.issubset(EXPECTED_CYCLES)

train dates 2025-01-01 to 2025-03-31 | cycles [np.int64(0), np.int64(1), np.int64(2)]
val   dates 2025-01-01 to 2025-03-31 | cycles [np.int64(0), np.int64(1), np.int64(2)]
test  dates 2025-01-01 to 2025-03-31 | cycles [np.int64(0), np.int64(1), np.int64(2)]


## 5. Scaling decision: nothing is scaled here, on purpose

XGBoost is tree-based, so monotonic rescaling cannot change which splits are chosen. The X_v2 columns were already scaled in feature engineering and are used exactly as they are. No scaler is fitted or applied in this notebook.

## 6. Final feature list (from `xgboost_feature_columns_final.json`) and leakage guard

In [5]:
with open(f"{ARTIFACT_DIR}/xgboost_feature_columns_final.json") as f:
    final_spec = json.load(f)
feature_cols = list(final_spec["feature_columns"])
assert len(feature_cols) == final_spec["n_features"] == 42

REMOVED = set(final_spec["removed"])
EXCLUDE = {"sim_id", "date", "billing_cycle_num"}
for name, X in [("train", X_train), ("val", X_val), ("test", X_test)]:
    leaked = [c for c in FORBIDDEN if c in X.columns]
    assert not leaked, f"{name}: label columns present: {leaked}"
    missing = [c for c in feature_cols if c not in X.columns]
    assert not missing, f"{name}: feature columns missing: {missing}"
assert not any(c in feature_cols for c in set(FORBIDDEN) | EXCLUDE | REMOVED)
assert not any(c.startswith("prophet_") for c in feature_cols)
assert "voice_utilization_ratio" in feature_cols

print(f"Final feature count: {len(feature_cols)}")
for i, c in enumerate(feature_cols, 1):
    print(f"{i:>3}. {c}")
print("\nRemoved from the 53 column model:", sorted(REMOVED))

Final feature count: 42
  1. total_download_mb
  2. total_upload_mb
  3. data_session_count
  4. data_duration_s
  5. hotspot_sessions
  6. public_apn_sessions
  7. unique_towers
  8. geo_flag_count
  9. peak_mb
 10. offpeak_mb
 11. total_mb
 12. roaming_sessions
 13. billed_mb
 14. billing_gap_mb
 15. billing_ratio
 16. offpeak_ratio
 17. public_apn_ratio
 18. upload_ratio
 19. bytes_per_session
 20. mean_session_duration_s
 21. hotspot_session_ratio
 22. has_geo_anomaly
 23. vol_rolling_7d_mean
 24. vol_rolling_7d_std
 25. vol_zscore_7d
 26. hotspot_sum_7d
 27. geo_sum_7d
 28. vol_rolling_14d_mean
 29. vol_rolling_14d_std
 30. vol_zscore_14d
 31. hotspot_sum_14d
 32. geo_sum_14d
 33. trend_slope_14d
 34. tier_field
 35. tier_manager
 36. tier_professional
 37. tier_support
 38. voice_utilization_ratio
 39. cohort_z_iso_anomaly_score
 40. cohort_z_total_mb
 41. cohort_z_vol_zscore_7d
 42. cohort_z_vol_zscore_14d

Removed from the 53 column model: ['days_to_depletion', 'projected_overa

## 7. Class imbalance: `scale_pos_weight`

In [6]:
ytr = y_train["is_anomalous"].values
n_pos, n_neg = int((ytr == 1).sum()), int((ytr == 0).sum())
scale_pos_weight = n_neg / n_pos
print(f"train positives={n_pos:,}  negatives={n_neg:,}")
print(f"anomaly rate         : {n_pos / len(ytr):.4%}")
print(f"scale_pos_weight     : {scale_pos_weight:.4f}")

train positives=1,292  negatives=61,708
anomaly rate         : 2.0508%
scale_pos_weight     : 47.7616


## 8. Fit XGBClassifier with early stopping on validation PR-AUC

In [7]:
if mlflow.active_run():
    mlflow.end_run()
run = mlflow.start_run(run_name="xgboost_triage_final_42col")

PARAMS = dict(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr", early_stopping_rounds=50,
    tree_method="hist", random_state=SEED, n_jobs=-1,
)

def fit_model():
    m = xgb.XGBClassifier(**PARAMS)
    m.fit(X_train[feature_cols], y_train["is_anomalous"].values,
          eval_set=[(X_val[feature_cols], y_val["is_anomalous"].values)], verbose=False)
    return m

model = fit_model()

best_iteration = int(model.best_iteration)
print(f"n_estimators requested : {PARAMS['n_estimators']}")
print(f"best_iteration (0-idx) : {best_iteration}  (trees used: {best_iteration + 1})")
print(f"best val aucpr         : {model.best_score:.4f}")

mlflow.log_params({
    "scale_pos_weight": round(scale_pos_weight, 4), "max_depth": PARAMS["max_depth"],
    "n_estimators": PARAMS["n_estimators"], "learning_rate": PARAMS["learning_rate"],
    "early_stopping_rounds": PARAMS["early_stopping_rounds"], "best_iteration": best_iteration,
    "subsample": PARAMS["subsample"], "colsample_bytree": PARAMS["colsample_bytree"],
    "seed": SEED, "n_features": len(feature_cols),
    "feature_list": "artifacts/xgboost_feature_columns_final.json",
    "gate_low": GATE_LOW, "gate_high": GATE_HIGH,
})

n_estimators requested : 500
best_iteration (0-idx) : 56  (trees used: 57)
best val aucpr         : 0.9253


## 9. Predicted probabilities and direction check

In [8]:
p_val  = model.predict_proba(X_val[feature_cols])[:, 1]
p_test = model.predict_proba(X_test[feature_cols])[:, 1]
yv, yt = y_val["is_anomalous"].values, y_test["is_anomalous"].values

rng = np.random.default_rng(SEED)
idx = rng.choice(len(p_val), size=min(5000, len(p_val)), replace=False)
corr = np.corrcoef(p_val[idx], yv[idx])[0, 1]
print(f"corr(prob, is_anomalous) on 5,000 val rows: {corr:.4f}")
print(f"mean prob | anomalous: {p_val[yv == 1].mean():.4f}   normal: {p_val[yv == 0].mean():.4f}")
print(f"max prob  | val: {p_val.max():.4f}   test: {p_test.max():.4f}")
assert corr > 0, "Probability is not positively correlated with anomalies"
print("Direction check passed: higher probability = more anomalous")

corr(prob, is_anomalous) on 5,000 val rows: 0.8865
mean prob | anomalous: 0.8766   normal: 0.0972
max prob  | val: 0.9733   test: 0.9733
Direction check passed: higher probability = more anomalous


## 10. Determinism check

Fit the same model a second time with the same data, seed and parameters, and compare the validation predictions element by element.

In [9]:
model_2 = fit_model()
p_val_2 = model_2.predict_proba(X_val[feature_cols])[:, 1]
DETERMINISTIC = bool(np.array_equal(p_val, p_val_2)) and int(model_2.best_iteration) == best_iteration
print(f"best_iteration run 1 / run 2 : {best_iteration} / {int(model_2.best_iteration)}")
print(f"max abs diff in val probs    : {np.abs(p_val - p_val_2).max():.3e}")
print(f"identical val predictions    : {np.array_equal(p_val, p_val_2)}")
assert DETERMINISTIC, "Two fits with the same seed gave different validation predictions"
mlflow.log_param("deterministic_refit", DETERMINISTIC)
del model_2, p_val_2

best_iteration run 1 / run 2 : 56 / 56
max abs diff in val probs    : 0.000e+00
identical val predictions    : True


## 11. Stage 3 gate

In [10]:
def gate(p):
    return np.where(p < GATE_LOW, "auto_clear", np.where(p > GATE_HIGH, "auto_flag", "escalated"))

BUCKETS = ["auto_clear", "escalated", "auto_flag"]
g_val, g_test = gate(p_val), gate(p_test)
for name, g in [("val", g_val), ("test", g_test)]:
    print(f"\n{name.upper()} gate buckets (n={len(g):,})")
    for b in BUCKETS:
        n = int((g == b).sum())
        print(f"  {b:<11} {n:>8,}  {n / len(g):>7.2%}")


VAL gate buckets (n=13,500)
  auto_clear     9,155   67.81%
  escalated      3,893   28.84%
  auto_flag        452    3.35%

TEST gate buckets (n=13,500)
  auto_clear     9,309   68.96%
  escalated      3,887   28.79%
  auto_flag        304    2.25%


## 12. Escalated band: baseline precision before CBR

In [11]:
def band_report(name, g, y):
    esc = g == "escalated"
    n_esc, a_esc = int(esc.sum()), int(y[esc].sum())
    total_anom = int(y.sum())
    print(f"\n{name.upper()}")
    print(f"  escalated rows            : {n_esc:,}")
    print(f"  anomalous in escalated    : {a_esc:,}  -> baseline precision {a_esc / n_esc if n_esc else float('nan'):.4f}")
    for b in BUCKETS:
        mask = g == b
        k = int(y[mask].sum())
        print(f"  anomalies in {b:<10}   : {k:>6,}  ({k / total_anom:.2%} of all anomalies)")
    flag = g == "auto_flag"
    clr = g == "auto_clear"
    print(f"  auto_flag precision       : {y[flag].mean() if flag.any() else float('nan'):.4f}")
    print(f"  auto_clear anomaly rate   : {y[clr].mean() if clr.any() else float('nan'):.4f}  (missed anomalies sit here)")
    return a_esc / n_esc if n_esc else float("nan")

esc_prec_val = band_report("val", g_val, yv)
esc_prec_test = band_report("test", g_test, yt)


VAL
  escalated rows            : 3,893
  anomalous in escalated    : 42  -> baseline precision 0.0108
  anomalies in auto_clear   :     32  (6.11% of all anomalies)
  anomalies in escalated    :     42  (8.02% of all anomalies)
  anomalies in auto_flag    :    450  (85.88% of all anomalies)
  auto_flag precision       : 0.9956
  auto_clear anomaly rate   : 0.0035  (missed anomalies sit here)

TEST
  escalated rows            : 3,887
  anomalous in escalated    : 49  -> baseline precision 0.0126
  anomalies in auto_clear   :     13  (3.60% of all anomalies)
  anomalies in escalated    :     49  (13.57% of all anomalies)
  anomalies in auto_flag    :    299  (82.83% of all anomalies)
  auto_flag precision       : 0.9836
  auto_clear anomaly rate   : 0.0014  (missed anomalies sit here)


## 13. Overall precision / recall / F1 at 0.5 and at the auto-flag threshold (0.90)

In [12]:
def overall(y, p, thr):
    pred = (p >= thr).astype(int)
    return dict(precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred, zero_division=0),
                f1=f1_score(y, pred, zero_division=0))

rows = []
for name, y, p in [("val", yv, p_val), ("test", yt, p_test)]:
    for thr in (0.5, GATE_HIGH):
        r = overall(y, p, thr); r.update(split=name, threshold=thr); rows.append(r)
    rows.append(dict(split=name, threshold="PR-AUC", precision=average_precision_score(y, p), recall=np.nan, f1=np.nan))
overall_df = pd.DataFrame(rows)[["split", "threshold", "precision", "recall", "f1"]]
print(overall_df.round(4).to_string(index=False))

split threshold  precision  recall     f1
  val       0.5     0.8887  0.8989 0.8937
  val       0.9     0.9956  0.8588 0.9221
  val    PR-AUC     0.9254     NaN    NaN
 test       0.5     0.8959  0.9058 0.9008
 test       0.9     0.9836  0.8283 0.8992
 test    PR-AUC     0.9297     NaN    NaN


## 14. Per-category recall and full-set precision (labels joined here, only for evaluation)

In [13]:
MISUSE_CATS = ["ghost_usage", "unauthorized_sharing", "data_reselling", "inactive_billing", "normal"]

eval_val  = X_val[["sim_id", "date"]].assign(is_anomalous=yv, misuse_category=y_val["misuse_category"].values, prob=p_val)
eval_test = X_test[["sim_id", "date"]].assign(is_anomalous=yt, misuse_category=y_test["misuse_category"].values, prob=p_test)

def category_table(ev, thr):
    # Recall within each category; precision over the FULL set: TP / (TP + FP), FP = flagged rows not in that category.
    flagged = (ev["prob"] >= thr).values
    out = []
    for cat in MISUSE_CATS:
        in_cat = (ev["misuse_category"] == cat).values
        hit = ~flagged if cat == "normal" else flagged
        tp = int((hit & in_cat).sum()); fp = int((hit & ~in_cat).sum())
        rec = tp / in_cat.sum() if in_cat.sum() else np.nan
        pre = tp / (tp + fp) if (tp + fp) else 0.0
        f1 = 2 * pre * rec / (pre + rec) if (pre + rec) else 0.0
        out.append(dict(category=cat, n=int(in_cat.sum()), precision=pre, recall=rec, f1=f1))
    return pd.DataFrame(out)

cat_tables = {}
for sname, ev in [("val", eval_val), ("test", eval_test)]:
    for thr in (0.5, GATE_HIGH):
        t = category_table(ev, thr)
        cat_tables[(sname, thr)] = t
        print(f"\n{sname.upper()} per-category at threshold {thr}")
        print(t.round(4).to_string(index=False))


VAL per-category at threshold 0.5
            category     n  precision  recall     f1
         ghost_usage   150     0.2642  0.9333 0.4118
unauthorized_sharing   217     0.3528  0.8618 0.5007
      data_reselling   101     0.1755  0.9208 0.2948
    inactive_billing    56     0.0962  0.9107 0.1741
              normal 12976     0.9959  0.9955 0.9957

VAL per-category at threshold 0.9
            category     n  precision  recall     f1
         ghost_usage   150     0.3097  0.9333 0.4651
unauthorized_sharing   217     0.3938  0.8203 0.5321
      data_reselling   101     0.1792  0.8020 0.2929
    inactive_billing    56     0.1128  0.9107 0.2008
              normal 12976     0.9943  0.9998 0.9971

TEST per-category at threshold 0.5
            category     n  precision  recall     f1
         ghost_usage    80     0.1973  0.9000 0.3236
unauthorized_sharing   235     0.5918  0.9191 0.7200
      data_reselling    24     0.0548  0.8333 0.1028
    inactive_billing    22     0.0521  0.8636 

## 15. Comparison against Isolation Forest and the cohort Z-score results

In [14]:
IF_RECALL  = {"ghost_usage": 0.1667, "unauthorized_sharing": 0.8295, "data_reselling": 0.3663, "inactive_billing": 0.8571}
Z_DELTA    = {"ghost_usage": 0.02, "data_reselling": 0.13, "unauthorized_sharing": 0.06, "inactive_billing": 0.08}

def compare(sname):
    t05 = cat_tables[(sname, 0.5)].set_index("category")
    t90 = cat_tables[(sname, GATE_HIGH)].set_index("category")
    rows = []
    for cat in IF_RECALL:
        rows.append(dict(
            category=cat,
            IF_recall=IF_RECALL[cat],
            IF_plus_Z_approx=IF_RECALL[cat] + Z_DELTA[cat],
            XGB_recall_at_0_5=t05.loc[cat, "recall"],
            XGB_recall_at_0_9=t90.loc[cat, "recall"],
            XGB_precision_at_0_5=t05.loc[cat, "precision"],
        ))
    df = pd.DataFrame(rows)
    df["delta_vs_IF"] = df["XGB_recall_at_0_5"] - df["IF_recall"]
    df["delta_vs_IF_plus_Z"] = df["XGB_recall_at_0_5"] - df["IF_plus_Z_approx"]
    return df

for sname in ("val", "test"):
    cmp_df = compare(sname)
    print(f"\n{sname.upper()}: recall comparison (XGBoost at the standard 0.5 threshold)")
    print(cmp_df.round(4).to_string(index=False))

print("\nNotes: IF recalls and Z-score deltas are the numbers already on record, copied from xgboost_triage.ipynb. "
      "IF_plus_Z_approx is an estimate, not a re-run.")


VAL: recall comparison (XGBoost at the standard 0.5 threshold)
            category  IF_recall  IF_plus_Z_approx  XGB_recall_at_0_5  XGB_recall_at_0_9  XGB_precision_at_0_5  delta_vs_IF  delta_vs_IF_plus_Z
         ghost_usage     0.1667            0.1867             0.9333             0.9333                0.2642       0.7666              0.7466
unauthorized_sharing     0.8295            0.8895             0.8618             0.8203                0.3528       0.0323             -0.0277
      data_reselling     0.3663            0.4963             0.9208             0.8020                0.1755       0.5545              0.4245
    inactive_billing     0.8571            0.9371             0.9107             0.9107                0.0962       0.0536             -0.0264

TEST: recall comparison (XGBoost at the standard 0.5 threshold)
            category  IF_recall  IF_plus_Z_approx  XGB_recall_at_0_5  XGB_recall_at_0_9  XGB_precision_at_0_5  delta_vs_IF  delta_vs_IF_plus_Z
         ghos

## 16. SHAP values on validation and summary plot

In [15]:
dval = xgb.DMatrix(X_val[feature_cols])

# Early stopping keeps every tree grown in the booster (best_iteration + 1 + patience trees),
# but predict_proba only uses trees 0..best_iteration. The raw margin and the SHAP values must
# use that same range, or SHAP explains a bigger model than the one that makes predictions.
TREES = (0, best_iteration + 1)
margin = model.get_booster().predict(dval, output_margin=True, iteration_range=TREES)   # raw log-odds
assert np.allclose(1 / (1 + np.exp(-margin)), p_val, atol=1e-6), "margin does not match predict_proba"

# Saved SHAP values: XGBoost native pred_contribs (exact TreeSHAP), limited to the same trees.
contrib = model.get_booster().predict(dval, pred_contribs=True, iteration_range=TREES)
sv, base_value = contrib[:, :-1], float(contrib[0, -1])
assert sv.shape == (len(X_val), len(feature_cols)), sv.shape
SHAP_SOURCE = "xgboost pred_contribs"

# Additivity check: base + sum(SHAP) must equal the model's raw margin (log-odds)
err = np.abs(base_value + sv.sum(axis=1) - margin).max()
print(f"trees explained: {TREES[1]} of {model.get_booster().num_boosted_rounds()} in the booster")
print(f"base value (log-odds): {base_value:.6f}")
print(f"pred_contribs additivity max abs error: {err:.2e}")
SHAP_ADDITIVE = bool(err < 1e-3)

# Cross-check only: shap.TreeExplainer also stops at best_iteration for this model.
te_sv = np.asarray(shap.TreeExplainer(model).shap_values(X_val[feature_cols]))
print(f"max abs diff shap.TreeExplainer vs pred_contribs: {np.abs(te_sv - sv).max():.2e}")
del te_sv
print("SHAP source:", SHAP_SOURCE)
mlflow.set_tags({"shap_iteration_range": f"0-{TREES[1]}", "shap_source": SHAP_SOURCE})

mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=feature_cols).sort_values(ascending=False)
top20 = mean_abs.head(20)[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top20.index, top20.values, color="#2f6db5")
ax.set_xlabel("mean |SHAP value| (log-odds)")
ax.set_title("XGBoost triage (42 columns): top 20 features by mean |SHAP|")
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/shap_summary.png", dpi=150)
plt.close(fig)

trees explained: 57 of 107 in the booster
base value (log-odds): 0.035506
pred_contribs additivity max abs error: 1.67e-06


max abs diff shap.TreeExplainer vs pred_contribs: 0.00e+00
SHAP source: xgboost pred_contribs


## 17. Save SHAP values for the escalated band (Stage 4 CBR input)

In [16]:
esc_mask = (p_val >= GATE_LOW) & (p_val <= GATE_HIGH)
shap_esc = sv[esc_mask]
esc_index = X_val.loc[esc_mask, ["sim_id", "date", "billing_cycle_num"]].reset_index(drop=True)
assert len(shap_esc) == len(esc_index)
assert shap_esc.shape[1] == 42

# Every escalated row: base + sum(SHAP) must equal logit(predict_proba) within 1e-4.
p_esc = p_val[esc_mask].astype(np.float64)
logit_err = np.abs(base_value + shap_esc.sum(axis=1) - np.log(p_esc / (1 - p_esc))).max()
print(f"escalated rows: base + sum(SHAP) vs logit(predict_proba) max abs error: {logit_err:.2e}")
SHAP_MATCHES_PROBA = bool(logit_err <= 1e-4)
assert SHAP_MATCHES_PROBA, "SHAP does not explain predict_proba on the escalated rows"

np.save(f"{ARTIFACT_DIR}/shap_values_escalated.npy", shap_esc)
esc_index.to_csv(f"{ARTIFACT_DIR}/shap_values_escalated_index.csv", index=False)
with open(f"{ARTIFACT_DIR}/shap_base_value.json", "w") as f:
    json.dump({"base_value_log_odds": base_value, "shap_source": SHAP_SOURCE, "iteration_range": list(TREES), "feature_cols": feature_cols}, f, indent=2)

print("shap_values_escalated.npy        :", shap_esc.shape, "(rows = escalated val rows, cols = feature_cols order)")
print("shap_values_escalated_index.csv  :", esc_index.shape)
print("Row i of the .npy matches row i of the CSV. SHAP units are log-odds.")

escalated rows: base + sum(SHAP) vs logit(predict_proba) max abs error: 1.18e-06
shap_values_escalated.npy        : (3893, 42) (rows = escalated val rows, cols = feature_cols order)
shap_values_escalated_index.csv  : (3893, 3)
Row i of the .npy matches row i of the CSV. SHAP units are log-odds.


## 18. Top 10 SHAP features

In [17]:
IF_TOP = ["hotspot_session_ratio", "hotspot_sessions", "hotspot_sum_7d", "hotspot_sum_14d", "billing_ratio", "billing_gap_mb"]
top10 = mean_abs.head(10)
print("Top 10 features by mean |SHAP|:")
for i, (f, v) in enumerate(top10.items(), 1):
    tag = "cohort_z" if f.startswith("cohort_z_") else "base"
    print(f"{i:>3}. {f:<32} {v:.4f}  [{tag}]{'  <- also in IF top features' if f in IF_TOP else ''}")

is_cohort = lambda f: f.startswith("cohort_z_")
total = mean_abs.sum()
share = {
    "cohort_z": mean_abs[[f for f in feature_cols if is_cohort(f)]].sum() / total,
    "IF_top_features": mean_abs[[f for f in IF_TOP if f in feature_cols]].sum() / total,
}
print("\nShare of total mean |SHAP|:")
for k, v in share.items():
    print(f"  {k:<16} {v:.1%}")
ranks = {f: int(i + 1) for i, f in enumerate(mean_abs.index)}
best_z = min((f for f in feature_cols if is_cohort(f)), key=ranks.get)
print(f"\nBest-ranked cohort-Z feature: {best_z} (rank {ranks[best_z]} of {len(feature_cols)})")
print(f"cohort_z columns in the top 10: {[f for f in top10.index if is_cohort(f)]}")

Top 10 features by mean |SHAP|:
  1. peak_mb                          0.9412  [base]
  2. cohort_z_iso_anomaly_score       0.3942  [cohort_z]
  3. hotspot_session_ratio            0.2972  [base]  <- also in IF top features
  4. offpeak_ratio                    0.2054  [base]
  5. offpeak_mb                       0.0997  [base]
  6. bytes_per_session                0.0980  [base]
  7. hotspot_sum_7d                   0.0936  [base]  <- also in IF top features
  8. hotspot_sessions                 0.0799  [base]  <- also in IF top features
  9. cohort_z_vol_zscore_14d          0.0727  [cohort_z]
 10. cohort_z_total_mb                0.0528  [cohort_z]

Share of total mean |SHAP|:
  cohort_z         18.4%
  IF_top_features  19.0%

Best-ranked cohort-Z feature: cohort_z_iso_anomaly_score (rank 2 of 42)
cohort_z columns in the top 10: ['cohort_z_iso_anomaly_score', 'cohort_z_vol_zscore_14d', 'cohort_z_total_mb']


## 19. Save model, feature list, and log to MLflow

In [18]:
joblib.dump(model, f"{ARTIFACT_DIR}/xgboost_triage.joblib")
with open(f"{ARTIFACT_DIR}/xgboost_feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

try:
    mlflow.xgboost.log_model(model, name="xgboost_triage")
except TypeError:
    mlflow.xgboost.log_model(model, artifact_path="xgboost_triage")

metrics = {
    "val_pr_auc": average_precision_score(yv, p_val), "test_pr_auc": average_precision_score(yt, p_test),
    "val_escalated_precision": esc_prec_val, "test_escalated_precision": esc_prec_test,
    "val_frac_escalated": float((g_val == "escalated").mean()), "test_frac_escalated": float((g_test == "escalated").mean()),
    "val_frac_auto_flag": float((g_val == "auto_flag").mean()), "test_frac_auto_flag": float((g_test == "auto_flag").mean()),
    "val_frac_auto_clear": float((g_val == "auto_clear").mean()), "test_frac_auto_clear": float((g_test == "auto_clear").mean()),
}
for name, y, p in [("val", yv, p_val), ("test", yt, p_test)]:
    for thr, tag in ((0.5, "0_5"), (GATE_HIGH, "0_9")):
        for k, v in overall(y, p, thr).items():
            metrics[f"{name}_{k}_at_{tag}"] = v
mlflow.log_metrics({k: float(v) for k, v in metrics.items() if not np.isnan(v)})
for fn in ["shap_summary.png", "shap_values_escalated.npy", "shap_values_escalated_index.csv",
           "shap_base_value.json", "xgboost_feature_columns.json", "xgboost_feature_columns_final.json"]:
    mlflow.log_artifact(f"{ARTIFACT_DIR}/{fn}")
RUN_ID = run.info.run_id
mlflow.end_run()
print("MLflow run:", RUN_ID, "| experiment:", EXPERIMENT_NAME)
print("Saved:", [f for f in sorted(os.listdir(ARTIFACT_DIR)) if f.startswith(("xgboost", "shap"))])

MLflow run: 72c7d016b9584def89647a49ed0d54bf | experiment: simguard-stage3-xgboost-final
Saved: ['shap_base_value.json', 'shap_summary.png', 'shap_values_escalated.npy', 'shap_values_escalated_index.csv', 'xgboost_feature_columns.json', 'xgboost_feature_columns_final.json', 'xgboost_triage.joblib']


## Sanity checks

In [19]:
checks = {}
checks["X_train/val/test exclude is_anomalous and misuse_category"] = all(
    not any(c in X.columns for c in FORBIDDEN) for X in (X_train, X_val, X_test))
finite = all(np.isfinite(X[feature_cols].to_numpy(dtype=float)).all() for X in (X_train, X_val, X_test))
checks["no NaN or inf in final feature columns"] = finite
s_tr, s_va, s_te = (set(X["sim_id"]) for X in (X_train, X_val, X_test))
checks["no SIM appears in more than one split"] = not (s_tr & s_va or s_tr & s_te or s_va & s_te)
with open(f"{ARTIFACT_DIR}/xgboost_feature_columns.json") as f:
    checks["xgboost_feature_columns.json equals the final 42 list"] = json.load(f) == final_spec["feature_columns"]
checks["saved model uses the final 42 columns"] = list(joblib.load(f"{ARTIFACT_DIR}/xgboost_triage.joblib").feature_names_in_) == feature_cols
checks["no Prophet or raw voice/SMS columns"] = not (set(feature_cols) & REMOVED)
checks["SHAP values additive (base + sum = model margin)"] = SHAP_ADDITIVE
checks["escalated rows: base + sum(SHAP) = logit(predict_proba) within 1e-4"] = SHAP_MATCHES_PROBA
checks["two fits give identical validation predictions"] = DETERMINISTIC

n_req = PARAMS["n_estimators"]
print(f"Early stopping: best_iteration={best_iteration} (trees used {best_iteration + 1}) vs n_estimators requested={n_req}")
early_flag = best_iteration + 1 < 30
never_flag = best_iteration + 1 >= n_req - PARAMS["early_stopping_rounds"]
if early_flag:
    print("  FLAG: early stopping triggered very early (< 30 trees). Possible overfitting risk or weak signal.")
if never_flag:
    print("  FLAG: early stopping effectively never triggered. n_estimators may be too low.")
checks["early stopping in a healthy range (not very early, not exhausted)"] = not (early_flag or never_flag)

print()
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print("\nALL SANITY CHECKS PASSED" if all(checks.values()) else "\nSOME SANITY CHECKS FAILED, see above")

Early stopping: best_iteration=56 (trees used 57) vs n_estimators requested=500

[PASS] X_train/val/test exclude is_anomalous and misuse_category
[PASS] no NaN or inf in final feature columns
[PASS] no SIM appears in more than one split
[PASS] xgboost_feature_columns.json equals the final 42 list
[PASS] saved model uses the final 42 columns
[PASS] no Prophet or raw voice/SMS columns
[PASS] SHAP values additive (base + sum = model margin)
[PASS] escalated rows: base + sum(SHAP) = logit(predict_proba) within 1e-4
[PASS] two fits give identical validation predictions
[PASS] early stopping in a healthy range (not very early, not exhausted)

ALL SANITY CHECKS PASSED
